# ASL Transformer Model Training for Sign0

Run this notebook in Google Colab to train a **Spatial-Temporal Transformer** model for sequence-based sign language recognition.

Outputs:
- `best_transformer_model.pth`
- `asl_transformer.onnx`

After training, copy `asl_transformer.onnx` into your repo's `backend/` folder.

## 1. Runtime Setup

In Colab, use **Runtime > Change runtime type > GPU** for faster training.

In [ ]:
!pip -q install onnx onnxruntime scikit-learn matplotlib tqdm

import os, sys, math, time, json, random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Configuration & Dataset Generation

We simulate temporal sequences since static images don't contain motion. Each sequence represents a gesture over time.

In [ ]:
LABELS = [chr(i) for i in range(ord('A'), ord('Z') + 1)]

def generate_sequence_dataset(num_samples_per_class=200, seq_len=16):
    X_sequences, y_labels = [], []
    for class_idx in range(26):
        base = np.random.uniform(-0.4, 0.4, size=(21, 3)).astype(np.float32)
        base[0] = [0, 0, 0] # Wrist
        for _ in range(num_samples_per_class):
            seq = []
            freq = np.random.uniform(0.5, 2.0)
            phase = np.random.uniform(0, math.pi)
            for t in range(seq_len):
                factor = math.sin(t * 0.2 * freq + phase) * 0.1
                frame_kp = base.copy()
                frame_kp[1:] += factor
                seq.append(frame_kp.reshape(63))
            X_sequences.append(np.array(seq, dtype=np.float32))
            y_labels.append(class_idx)
    return np.array(X_sequences), np.array(y_labels)

print("Generating Temporal Sequence Dataset...")
X_seq, y = generate_sequence_dataset(num_samples_per_class=300, seq_len=16)
print(f"Sequence Dataset Shape: X={X_seq.shape}, y={y.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    X_seq, y, test_size=0.2, random_state=SEED, stratify=y
)

class SignSequenceDataset(Dataset):
    def __init__(self, X_seq, y):
        self.X_seq = torch.tensor(X_seq, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X_seq[idx], self.y[idx]

train_loader = DataLoader(SignSequenceDataset(X_train, y_train), batch_size=64, shuffle=True)
test_loader = DataLoader(SignSequenceDataset(X_test, y_test), batch_size=64, shuffle=False)
print("Dataset loaders ready!")

## 3. Spatial-Temporal Sign Transformer Model

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class SpatialTemporalSignTransformer(nn.Module):
    def __init__(self, input_dim=63, d_model=128, nhead=4, num_layers=3, num_classes=26, max_len=60, dropout=0.2):
        super().__init__()
        self.input_projection = nn.Sequential(
            nn.Linear(input_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.pos_encoder = PositionalEncoding(d_model, max_len=max_len)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 2,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        h = self.input_projection(x)
        h = self.pos_encoder(h)
        out = self.transformer_encoder(h)
        pooled = out.mean(dim=1)
        return self.head(pooled)

model = SpatialTemporalSignTransformer(input_dim=63, d_model=128, nhead=4, num_layers=3, num_classes=26, max_len=30).to(device)
print(model)

## 4. Training Loop

In [ ]:
EPOCHS = 35
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

best_val_acc = 0.0
best_path = "best_transformer_model.pth"

print("Starting Transformer Training...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        logits = model(bx)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * bx.size(0)
        correct += (logits.argmax(1) == by).sum().item()
        total += bx.size(0)
        
    train_acc = correct / total
    train_loss = total_loss / total
    
    model.eval()
    vcorrect, vtotal, vloss = 0, 0, 0.0
    with torch.no_grad():
        for bx, by in test_loader:
            bx, by = bx.to(device), by.to(device)
            logits = model(bx)
            loss = criterion(logits, by)
            vloss += loss.item() * bx.size(0)
            vcorrect += (logits.argmax(1) == by).sum().item()
            vtotal += bx.size(0)
            
    val_acc = vcorrect / vtotal
    val_loss = vloss / vtotal
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_path)
        
    if epoch % 5 == 0 or epoch == EPOCHS:
        print(f"Epoch {epoch:02d}/{EPOCHS:02d} | Train Acc: {train_acc*100:5.2f}% (Loss: {train_loss:.4f}) | Val Acc: {val_acc*100:5.2f}% (Loss: {val_loss:.4f}) | Best: {best_val_acc*100:5.2f}%")

print("Training Complete! Best Validation Accuracy:", best_val_acc)

## 5. Export ONNX and Download

Exports the `asl_transformer.onnx` file ready for the FastAPI backend.

In [ ]:
model.load_state_dict(torch.load(best_path, map_location=device))
model.eval().cpu()

dummy_input = torch.randn(1, 16, 63, dtype=torch.float32)
onnx_path = "asl_transformer.onnx"

try:
    torch.onnx.export(
        model,
        dummy_input,
        onnx_path,
        input_names=["sequence_keypoints"],
        output_names=["logits"],
        dynamic_axes={
            "sequence_keypoints": {0: "batch_size", 1: "seq_len"},
            "logits": {0: "batch_size"}
        },
        opset_version=14,
        do_constant_folding=True
    )
    print("ONNX export successful:", onnx_path)
except Exception as e:
    print(f"ONNX opset 14 export failed: {e}. Trying opset 12...")
    torch.onnx.export(
        model,
        dummy_input,
        onnx_path,
        input_names=["sequence_keypoints"],
        output_names=["logits"],
        dynamic_axes={
            "sequence_keypoints": {0: "batch_size", 1: "seq_len"},
            "logits": {0: "batch_size"}
        },
        opset_version=12,
    )
    print("Legacy ONNX export successful:", onnx_path)

try:
    from google.colab import files
    files.download(onnx_path)
    files.download(best_path)
except ImportError:
    print("Not in Colab. Files saved locally.")